# Cluster copies: one copy of lakefed_ingest per app job with its own cluster

The operations app's Edit view sets an app job's cluster (driver type, worker type, worker count) in its schedule's
`options.cluster`. A `run_job` task or run-now can't change a job's cluster, so such an app job runs a copy of
`lakefed_ingest` with only those settings changed (`cluster_copies.py`). This job makes, updates and deletes those
copies and records each copy's id in `options.ingest_copy_job_id`, where the app's sync picks it up.

- **Identity:** it runs as the deployer, like `lakefed_ingest`, so a copy (owned by the deployer and running as the
  deployer) has exactly the ingestion job's rights. The app's service principal can't make the copies itself: a job
  it created would run as the service principal.
- **Access:** each copy gets the ingestion job's own grants, plus Can Manage for the app's service principal
  (`app_service_principal`, passed by the app), so Run now, Delete and the app's checks work on it as on the
  ingestion job.
- **Which schedules:** `schedule_id` (blank = every schedule). Each run copies the bundle's job again, so a
  redeploy reaches the copies on the next run.

In [ ]:
import sys, os

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
nb_dir = os.path.dirname(ctx.notebookPath().get())
if ("/Workspace" + nb_dir) not in sys.path:
    sys.path.append("/Workspace" + nb_dir)

from databricks.sdk import WorkspaceClient
from state_store import StateStore
import cluster_copies as cc


def widget(name, default=""):
    dbutils.widgets.text(name, default)
    return dbutils.widgets.get(name).strip()


ingest_job_id = int(widget("ingest_job_id", "0"))
schedule_id = widget("schedule_id")
app_principal = widget("app_service_principal") or None
db_instance = widget("db_instance", "lakefed-ingest")
db_name = widget("db_name", "lakefed_ingest")
assert ingest_job_id > 0, "ingest_job_id is required: the bundle's lakefed_ingest job"
owner = f"{db_instance}/{db_name}"

In [ ]:
w = WorkspaceClient()
deployer = w.current_user.me().user_name
bundle_job = w.jobs.get(ingest_job_id).settings.as_dict()
bundle_acl = [entry.as_dict() for entry in (w.permissions.get("jobs", str(ingest_job_id)).access_control_list or [])]
store = StateStore.connect_lakebase(db_instance, db_name)
with store.connection.cursor() as cur:
    cur.execute("select id, name, options, deleted_at from lakefed_ingest.schedule"
                + (" where id = %s" if schedule_id else ""), (int(schedule_id),) if schedule_id else ())
    columns = [d.name for d in cur.description]
    schedules = [dict(zip(columns, row)) for row in cur.fetchall()]


def our_copy(job_id, sid):
    """The copy a schedule records, if it still exists and is ours (made for this schedule and this app's database)."""
    try:
        tags = w.jobs.get(int(job_id)).settings.tags or {}
    except Exception:
        return None          # deleted outside the app: made again below
    return int(job_id) if tags.get(cc.TAG_SCHEDULE) == str(sid) and tags.get(cc.TAG_OWNER) == owner else None


copies = {}
for s in schedules:
    recorded = (s["options"] or {}).get("ingest_copy_job_id")
    if recorded and our_copy(recorded, s["id"]):
        copies[str(s["id"])] = int(recorded)
actions = cc.plan(schedules, copies, bundle_job, owner)
print(f"{len(schedules)} schedule(s), {len(copies)} copy(ies) found: "
      + (", ".join(f"{a['action']} for schedule {a['schedule_id']}" for a in actions) or "nothing to do"))

In [ ]:
def record(sid, job_id):
    with store.connection.cursor() as cur:
        if job_id:
            cur.execute("update lakefed_ingest.schedule set options = jsonb_set(coalesce(options, '{}'::jsonb), "
                        "'{ingest_copy_job_id}', to_jsonb(%s::bigint)) where id = %s", (job_id, sid))
        else:
            cur.execute("update lakefed_ingest.schedule set options = options - 'ingest_copy_job_id' where id = %s",
                        (sid,))


for action in actions:
    sid = action["schedule_id"]
    if action["action"] == "delete":
        w.jobs.delete(action["job_id"])
        record(sid, None)
        print(f"schedule {sid}: deleted copy {action['job_id']} (back to the bundle's job)")
        continue
    if action["action"] == "create":
        job_id = int(w.api_client.do("POST", "/api/2.2/jobs/create", body=action["settings"])["job_id"])
    else:
        job_id = action["job_id"]
        w.api_client.do("POST", "/api/2.2/jobs/reset", body={"job_id": job_id, "new_settings": action["settings"]})
    w.api_client.do("PUT", f"/api/2.0/permissions/jobs/{job_id}",
                    body={"access_control_list": cc.copy_permissions(bundle_acl, app_principal, deployer)})
    record(sid, job_id)
    cluster = cc.cluster_of(next(s for s in schedules if s["id"] == sid))
    print(f"schedule {sid}: {action['action']}d copy {job_id} with {cluster}")
store.close()